# SE3082 Parallel Computing - Lab 09
Student: IT24103467

Run each cell in order on a Colab GPU runtime. The source files are also in the GitHub repository.


In [ ]:
!nvidia-smi
!nvcc --version
!nvidia-smi --query-gpu=name,compute_cap --format=csv


The commands below target a T4 (`sm_75`). If Colab provides another GPU, change `-arch=sm_75` to match its compute capability.


In [ ]:
%%writefile cuda_utils.cuh
#ifndef CUDA_UTILS_CUH
#define CUDA_UTILS_CUH

#include <cuda_runtime.h>
#include <cstdio>
#include <cstdlib>

#define CHECK(call) do { \
    cudaError_t result = (call); \
    if (result != cudaSuccess) { \
        fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__, \
                cudaGetErrorString(result)); \
        exit(EXIT_FAILURE); \
    } \
} while (0)

static void *host_alloc(size_t bytes) {
    void *ptr = malloc(bytes);
    if (ptr == nullptr) {
        fprintf(stderr, "Host allocation failed for %zu bytes\n", bytes);
        exit(EXIT_FAILURE);
    }
    return ptr;
}

static void random_ints(int *values, size_t count) {
    for (size_t i = 0; i < count; ++i)
        values[i] = rand() % 100;
}

#endif


## Exercise 1 - GPU setup


In [ ]:
%%writefile hello.cu
#include "cuda_utils.cuh"

__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main() {
    hello<<<1, 4>>>();
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 hello.cu -o hello && ./hello


## Exercise 2 - Add two numbers


In [ ]:
%%writefile ex2_add.cu
#include "cuda_utils.cuh"

__global__ void add(const int *a, const int *b, int *c) {
    *c = *a + *b;
}

int main() {
    const int a = 2, b = 7;
    int c = 0;
    int *d_a, *d_b, *d_c;
    CHECK(cudaMalloc((void **)&d_a, sizeof(int)));
    CHECK(cudaMalloc((void **)&d_b, sizeof(int)));
    CHECK(cudaMalloc((void **)&d_c, sizeof(int)));
    CHECK(cudaMemcpy(d_a, &a, sizeof(int), cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, &b, sizeof(int), cudaMemcpyHostToDevice));
    add<<<1, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaMemcpy(&c, d_c, sizeof(int), cudaMemcpyDeviceToHost));
    printf("Result is %d\n", c);
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));
    return c == 9 ? 0 : 1;
}


In [ ]:
!nvcc -arch=sm_75 ex2_add.cu -o ex2_add && ./ex2_add


## Exercise 3 - 512 blocks


In [ ]:
%%writefile ex3_blocks.cu
#include "cuda_utils.cuh"

#define N 512

__global__ void add_blocks(const int *a, const int *b, int *c) {
    int i = blockIdx.x;
    c[i] = a[i] + b[i];
}

int main() {
    const size_t bytes = N * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, N);
    random_ints(b, N);
    int *d_a, *d_b, *d_c;
    CHECK(cudaMalloc((void **)&d_a, bytes));
    CHECK(cudaMalloc((void **)&d_b, bytes));
    CHECK(cudaMalloc((void **)&d_c, bytes));
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    add_blocks<<<N, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (int i = 0; i < N; ++i) {
        printf("%d + %d = %d\n", a[i], b[i], c[i]);
        errors += c[i] != a[i] + b[i];
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);
    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_blocks.cu -o ex3_blocks && ./ex3_blocks


## Exercise 4 - 512 threads


In [ ]:
%%writefile ex4_threads.cu
#include "cuda_utils.cuh"

#define N 512

__global__ void add_threads(const int *a, const int *b, int *c) {
    int i = threadIdx.x;
    c[i] = a[i] + b[i];
}

int main() {
    const size_t bytes = N * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, N);
    random_ints(b, N);
    int *d_a, *d_b, *d_c;
    CHECK(cudaMalloc((void **)&d_a, bytes));
    CHECK(cudaMalloc((void **)&d_b, bytes));
    CHECK(cudaMalloc((void **)&d_c, bytes));
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    add_threads<<<1, N>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));
    int errors = 0;
    for (int i = 0; i < N; ++i) {
        printf("%d) %d + %d = %d\n", i, a[i], b[i], c[i]);
        errors += c[i] != a[i] + b[i];
    }
    printf("Verification: %d mismatches out of %d\n", errors, N);
    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_threads.cu -o ex4_threads && ./ex4_threads


## Exercise 5 - Ten-million-element vector product


In [ ]:
%%writefile ex5_vecmul.cu
#include "cuda_utils.cuh"

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vec_mul(const int *a, const int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n)
        c[i] = a[i] * b[i];
}

int main() {
    const size_t bytes = (size_t)N * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, N);
    random_ints(b, N);
    int *d_a, *d_b, *d_c;
    CHECK(cudaMalloc((void **)&d_a, bytes));
    CHECK(cudaMalloc((void **)&d_b, bytes));
    CHECK(cudaMalloc((void **)&d_c, bytes));
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;
    printf("Launching %d blocks x %d threads\n", blocks, THREADS_PER_BLOCK);
    vec_mul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (int i = N - 1000; i < N; ++i)
        printf("%d) %d x %d = %d\n", i, a[i], b[i], c[i]);
    int errors = 0;
    for (int i = 0; i < N; ++i)
        errors += c[i] != a[i] * b[i];
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul.cu -o ex5_vecmul && ./ex5_vecmul


## Exercise 6 - 10,000 by 10,000 element-wise matrix product


In [ ]:
%%writefile ex6_matrix.cu
#include "cuda_utils.cuh"

#define ROWS 10000
#define COLS 10000

/* Element-wise product, not matrix multiplication. */
__global__ void mul_elem(const int *a, const int *b, int *c,
                         int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (row < rows && col < cols) {
        size_t idx = (size_t)row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

int main() {
    const size_t count = (size_t)ROWS * COLS;
    const size_t bytes = count * sizeof(int);
    int *a = (int *)host_alloc(bytes), *b = (int *)host_alloc(bytes);
    int *c = (int *)host_alloc(bytes);
    random_ints(a, count);
    random_ints(b, count);
    int *d_a, *d_b, *d_c;
    CHECK(cudaMalloc((void **)&d_a, bytes));
    CHECK(cudaMalloc((void **)&d_b, bytes));
    CHECK(cudaMalloc((void **)&d_c, bytes));
    CHECK(cudaMemcpy(d_a, a, bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));

    dim3 threads_per_block(32, 16);
    dim3 blocks((COLS + threads_per_block.x - 1) / threads_per_block.x,
                (ROWS + threads_per_block.y - 1) / threads_per_block.y);
    printf("Grid: %u x %u blocks\n", blocks.x, blocks.y);
    mul_elem<<<blocks, threads_per_block>>>(d_a, d_b, d_c, ROWS, COLS);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    CHECK(cudaMemcpy(c, d_c, bytes, cudaMemcpyDeviceToHost));

    for (size_t idx = count - 1000; idx < count; ++idx)
        printf("C[%zu][%zu] = %d x %d = %d\n",
               idx / COLS, idx % COLS, a[idx], b[idx], c[idx]);
    size_t errors = 0;
    for (size_t idx = 0; idx < count; ++idx)
        errors += c[idx] != a[idx] * b[idx];
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return errors ? 1 : 0;
}


In [ ]:
!nvcc -arch=sm_75 ex6_matrix.cu -o ex6_matrix && ./ex6_matrix
